# CatalogGuard — short analytical walkthrough
Run the CLI pipeline first. Select the project .venv kernel and run from the project root. These cells read existing results; they do not tune on test labels.

In [ ]:
import json
from pathlib import Path
import pandas as pd
root = Path.cwd()
assert (root / 'catalogguard').is_dir(), 'Run from the project root'
audit = json.loads((root / 'reports/data_audit.json').read_text())
pd.DataFrame(audit['split_counts']).T

## 1. Did the source splits overlap?
Compare the overlap audit with the custom split counts. Explain exclusions and why prevalence differs across partitions. Do not equate 8,428 source pairs with the retained training size.

In [ ]:
audit['original_split_listing_overlap'], audit['conflicting_pair_ids_excluded'], audit['cross_partition_pairs_excluded']

## 2. Did images add value?
Use validation comparisons, keeping the test set for final reporting. A larger model is not automatically better.

In [ ]:
models = pd.read_csv(root / 'reports/validation_models.csv')
models[['model', 'average_precision', 'precision', 'recall', 'f1']].sort_values('average_precision', ascending=False)

## 3. Which errors remain?
Final error analysis is descriptive. Any changes motivated by this analysis need a new independent test set before claiming an unbiased improvement.

In [ ]:
pred = pd.read_csv(root / 'reports/test_predictions.csv')
pd.crosstab(pred.label, pred.prediction, rownames=['Actual match'], colnames=['Predicted match'])

## 4. What is the review trade-off?
Suggestion precision applies only to selected pairs. Coverage and the selection threshold must be reported with it.

In [ ]:
report = json.loads((root / 'reports/test_metrics.json').read_text())
report['suggestions']

## 5. MongoDB aggregation on the separate larger catalog
Run abo-scale first. This is metadata analysis, not labelled duplicate-model validation.

In [ ]:
from catalogguard.store import database
db = database()
pd.DataFrame(list(db.abo_listings.aggregate([
    {'$unwind': '$product_type'},
    {'$group': {'_id': '$product_type.value', 'count': {'$sum': 1}}},
    {'$sort': {'count': -1}}, {'$limit': 10}
])))